  #                                                                       # *****DATA PRE PROCESSING*****

# ***Demography***

In [343]:
#Importing all the Necessary Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


import warnings
warnings.simplefilter("ignore", UserWarning)

In [344]:
from pathlib import Path
data_folder = Path(
    r"C:\Users\vella\Desktop\Numpy Ninja\Team3_PyCoders_PythonHackathon_SEP2026\Python_Hackathon_Sep_2026\Python_Hackathon_Sep_2026\cardiac_failure")

### **1. Read the CSV file and inspect the data. This confirms the file loaded correctly and shows missing values before you change anything**

In [345]:
# Read the demography CSV File and inspect the data 

dfDEMO = pd.read_csv(data_folder/"demography.csv")
df_original = dfDEMO.copy()

print("Rows and columns:", dfDEMO.shape)
display(dfDEMO.head())
dfDEMO.info()

print("\nMissing values:")
display(dfDEMO.isna().sum())

print(
    "Duplicate patient IDs:",
    dfDEMO["inpatient_number"].duplicated().sum()
)



Rows and columns: (2009, 7)


,inpatient_number,gender,weight,height,bmi,occupation,agecat
0,5,NaN,NaN,NaN,46.000000,NaN,NaN
1,827040,Female,50.0,1.45,23.781213,NaN,69-79
2,857781,Male,50.0,1.64,18.590125,UrbanResident,69-79
3,743087,Female,51.0,1.63,19.195303,UrbanResident,69-79
4,866418,Male,70.0,1.70,24.221453,farmer,59-69


<class 'pandas.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   inpatient_number  2009 non-null   int64  
 1   gender            2008 non-null   str    
 2   weight            2008 non-null   float64
 3   height            2008 non-null   float64
 4   bmi               2009 non-null   float64
 5   occupation        1981 non-null   str    
 6   agecat            2008 non-null   str    
dtypes: float64(3), int64(1), str(3)
memory usage: 110.0 KB

Missing values:


inpatient_number     0
gender               1
weight               1
height               1
bmi                  0
occupation          28
agecat               1
dtype: int64

Duplicate patient IDs: 0


### **2. Removed the empty record. Patient ID 5 has no gender, weight, height,occupation, or age category. Its BMI value alone is not enuogh to use the record for demographic analysis. So this removes one row**


In [346]:
dfDEMO = dfDEMO.dropna(
    subset=["gender", "weight", "height", "occupation", "agecat"],
    how="all"
).copy()

### **3. Renamed column names for best readability and consistency across datasets for less confusion during analysis**

In [347]:
dfDEMO = dfDEMO.rename(columns={
    "inpatient_number": "patient_id",
    "agecat" : "age_category"
})

### **4. Removed extra spaces from the text columns so values such as "Male" and " Male " are treated as the same category. I also make the occupation names consistent and easier to read. This prevents one category from appearing under different labels in charts and counts.**

In [348]:
print(dfDEMO.columns.tolist())

text_columns = ["gender", "occupation", "age_category"]

for column in text_columns:
    dfDEMO[column] = dfDEMO[column].astype("string").str.strip()

dfDEMO["occupation"] = dfDEMO["occupation"].replace({
    "UrbanResident": "Urban Resident",
    "farmer": "Farmer",
    "worker": "Worker"
})

dfDEMO["occupation"] = dfDEMO["occupation"].fillna("Unknown")



['patient_id', 'gender', 'weight', 'height', 'bmi', 'occupation', 'age_category']


### **5. Impossible measurements, Weight values of zero or less and height values below 1.0 are flagged for review. The code marks those rows in measurement and changes the flagged values to missing so they do not affect BMI calculations. It then counts how many records were flagged.***

In [349]:
dfDEMO["measurement"] = pd.Series(False, index=dfDEMO.index)

invalid_weight = dfDEMO["weight"] <= 0
invalid_height = dfDEMO["height"] < 1.0

dfDEMO.loc[invalid_weight | invalid_height, "measurement"] = True

dfDEMO.loc[invalid_weight, "weight"] = np.nan
dfDEMO.loc[invalid_height, "height"] = np.nan

print("Records with measurement issues:", dfDEMO["measurement"].sum())

Records with measurement issues: 7


### **6. Recalculated BMI from Valid measurements. The orginial BMI values match the recorded weights and heights, including the incorrect heights and zero weights, Recalculating after flagging those measurements makes BMI missing for the seven affected records**

In [350]:
dfDEMO["bmi_original"] = dfDEMO["bmi"].round(2)

dfDEMO["bmi"] = dfDEMO["weight"] / (dfDEMO["height"] ** 2)
dfDEMO["bmi"] = dfDEMO["bmi"].round(2)


### **7. Removed Unrealistic BMI values and validated age category consistency**

In [351]:
dfDEMO["bmi"] = pd.to_numeric(dfDEMO["bmi"], errors="coerce")
dfDEMO.loc[(dfDEMO["bmi"] < 15) | (dfDEMO["bmi"] > 60), "bmi"] = np.nan
dfDEMO["age_category"] = dfDEMO["age_category"].astype(str).str.strip()
dfDEMO["age_category"] = dfDEMO["age_category"].str.title()

### **8. Reviewed the cleaned data to make sure it is ready for analysis. I check the number of rows and unique patients, see which values are still missing, review the weight, height, and BMI ranges, and count each category. Finally, I display the records flagged for measurement issues so I can inspect them before using them in calculations.**

In [352]:
print("Rows:", len(dfDEMO))
print("Unique patient IDs:", dfDEMO["patient_id"].nunique())
print("\nMissing values:")
print(dfDEMO.isna().sum())

print("\nNumeric summary:")
print(dfDEMO[["weight", "height", "bmi"]].describe())

print("\nCategory counts:")
for column in ["gender", "occupation", "age_category"]:
    print(f"\n{column}")
    print(dfDEMO[column].value_counts(dropna=False))

print("\nRecords needing measurement review:")
print(
    dfDEMO.loc[
        dfDEMO["measurement"],
        ["patient_id", "weight", "height", "bmi_original"]
    ]
)

Rows: 2008
Unique patient IDs: 2008

Missing values:
patient_id       0
gender           0
weight           3
height           4
bmi             39
occupation       0
age_category     0
measurement      0
bmi_original     0
dtype: int64

Numeric summary:
            weight       height          bmi
count  2005.000000  2004.000000  1969.000000
mean     52.562244     1.570110    21.407984
std      10.713048     0.081954     3.732060
min       8.000000     1.200000    15.070000
25%      45.000000     1.500000    18.670000
50%      50.000000     1.560000    20.810000
75%      60.000000     1.620000    23.440000
max     115.000000     1.830000    39.110000

Category counts:

gender
gender
Female    1163
Male       845
Name: count, dtype: Int64

occupation
occupation
Urban Resident    1670
Farmer             198
Others              89
Unknown             27
Worker              17
Officer              7
Name: count, dtype: Int64

age_category
age_category
69-79     715
79-89     646
59-69    

### **9. Save analysis and reviewing the files**

In [353]:
dfDEMO.to_csv(data_folder / "demography_clean.csv", index=False)

#                                                          ***patienthistory***

### **1. Load the CSV File and check the size. column types, missing values, and patient IDs before changing anything**

In [354]:
dfPH = pd.read_csv(data_folder/"patienthistory.csv")
df_original = dfPH.copy()

print("Rows and columns:", dfPH.shape)
display(dfPH.head())
dfPH.info()

print("\nMissing values:")
display(dfPH.isna().sum())

print(
    "Duplicate patient IDs:",
    dfPH["inpatient_number"].duplicated().sum()
)


Rows and columns: (2008, 17)


,inpatient_number,cerebrovascular_disease,dementia,chronic_obstructive_pulmonary_disease,connective_tissue_disease,peptic_ulcer_disease,diabetes,moderate_to_severe_chronic_kidney_disease,hemiplegia,leukemia,malignant_lymphoma,solid_tumor,liver_disease,aids,cci_score,type_ii_respiratory_failure,acute_renal_failure
0,857781,0,0,1,0,0.0,1,0.0,0,0,0,0,0.0,0,2.0,nontypeii,0
1,743087,0,0,0,0,0.0,0,0.0,0,0,0,0,0.0,0,0.0,nontypeii,0
2,866418,0,0,0,0,0.0,0,0.0,0,0,0,0,0.0,0,0.0,nontypeii,0
3,775928,0,0,1,0,0.0,0,1.0,0,0,0,0,0.0,0,2.0,nontypeii,0
4,810128,0,0,0,0,0.0,0,0.0,0,0,0,0,0.0,0,0.0,nontypeii,0


<class 'pandas.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 17 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   inpatient_number                           2008 non-null   int64  
 1   cerebrovascular_disease                    2008 non-null   int64  
 2   dementia                                   2008 non-null   int64  
 3   chronic_obstructive_pulmonary_disease      2008 non-null   int64  
 4   connective_tissue_disease                  2008 non-null   int64  
 5   peptic_ulcer_disease                       2006 non-null   float64
 6   diabetes                                   2008 non-null   int64  
 7   moderate_to_severe_chronic_kidney_disease  2006 non-null   float64
 8   hemiplegia                                 2008 non-null   int64  
 9   leukemia                                   2008 non-null   int64  
 10  malignant_lymphoma                 

inpatient_number                             0
cerebrovascular_disease                      0
dementia                                     0
chronic_obstructive_pulmonary_disease        0
connective_tissue_disease                    0
peptic_ulcer_disease                         2
diabetes                                     0
moderate_to_severe_chronic_kidney_disease    2
hemiplegia                                   0
leukemia                                     0
malignant_lymphoma                           0
solid_tumor                                  0
liver_disease                                1
aids                                         0
cci_score                                    5
type_ii_respiratory_failure                  0
acute_renal_failure                          0
dtype: int64

Duplicate patient IDs: 0


### ***2. Renamed the column name inpatient_number to patient_id***

In [355]:
dfPH = dfPH.rename(columns={
    "inpatient_number": "patient_id",
    "type_ii_respiratory_failure": "respiratory_failure_type2",
    "moderate_to_severe_chronic_kidney_disease": "mod_severe_ckd",
    "chronic_obstructive_pulmonary_disease": "copd"
})

### ***3. Condition should check only 0,1, or missing values reviewing unexpected values before converting Types***

In [356]:
condition_columns = [
    "cerebrovascular_disease",
    "dementia",
    "copd",
    "connective_tissue_disease",
    "peptic_ulcer_disease",
    "diabetes",
    "mod_severe_ckd",
    "hemiplegia",
    "leukemia",
    "malignant_lymphoma",
    "solid_tumor",
    "liver_disease",
    "aids",
    "acute_renal_failure",
]

for column in condition_columns:
    print(f"\n{column}")
    print(dfPH[column].value_counts(dropna=False))


cerebrovascular_disease
cerebrovascular_disease
0    1858
1     150
Name: count, dtype: int64

dementia
dementia
0    1893
1     115
Name: count, dtype: int64

copd
copd
0    1775
1     233
Name: count, dtype: int64

connective_tissue_disease
connective_tissue_disease
0    2004
1       4
Name: count, dtype: int64

peptic_ulcer_disease
peptic_ulcer_disease
0.0    1961
1.0      45
NaN       2
Name: count, dtype: int64

diabetes
diabetes
0    1542
1     466
Name: count, dtype: int64

mod_severe_ckd
mod_severe_ckd
0.0    1532
1.0     474
NaN       2
Name: count, dtype: int64

hemiplegia
hemiplegia
0    1996
1      12
Name: count, dtype: int64

leukemia
leukemia
0    2008
Name: count, dtype: int64

malignant_lymphoma
malignant_lymphoma
0    2007
1       1
Name: count, dtype: int64

solid_tumor
solid_tumor
0    1969
1      39
Name: count, dtype: int64

liver_disease
liver_disease
0.0    1923
1.0      84
NaN       1
Name: count, dtype: int64

aids
aids
0    2004
1       4
Name: count, dtype:

### ***4. Consistent text labels prevent the same category from appearing under different names in charts***

In [357]:
dfPH["respiratory_failure_type2"] = (
    dfPH["respiratory_failure_type2"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print(dfPH["respiratory_failure_type2"].value_counts(dropna=False))

respiratory_failure_type2
nontypeii    1894
typeii        114
Name: count, dtype: Int64


### ***5. cci_score can be used to compare groups of patients, so check its range and missing values. Do not invent scores for patients whose values are missing***

In [358]:
dfPH["cci_score"] = pd.to_numeric(
    dfPH["cci_score"],
    errors="coerce"
).astype("Int64")

print(dfPH["cci_score"].value_counts(dropna=False).sort_index())
print("Missing CCI scores:", dfPH["cci_score"].isna().sum())
dfPH["cci_score"].describe()

cci_score
0        56
1       770
2       699
3       368
4        94
5        15
6         1
<NA>      5
Name: count, dtype: Int64
Missing CCI scores: 5


count      2003.0
mean     1.861707
std      0.961469
min           0.0
25%           1.0
50%           2.0
75%           2.0
max           6.0
Name: cci_score, dtype: Float64

### ***6. Checking the cleaned file and saving the file. To confirm that cleaning did not accidentally remove patients or create duplicate IDs before joining this file to the other datasets***

In [359]:
print("Original rows:", len(df_original))
print("Cleaned rows:", len(dfPH))
print("Unique patient IDs:", dfPH["patient_id"].nunique())
print("Duplicate patient IDs:", dfPH["patient_id"].duplicated().sum())

print("\nRemaining missing values:")
display(dfPH.isna().sum())

assert len(dfPH) == 2008
assert dfPH["patient_id"].is_unique


dfPH.to_csv(
    data_folder / "patienthistory_clean.csv",
    index=False
)

print("Saved patienthistory_clean.csv")

Original rows: 2008
Cleaned rows: 2008
Unique patient IDs: 2008
Duplicate patient IDs: 0

Remaining missing values:


patient_id                   0
cerebrovascular_disease      0
dementia                     0
copd                         0
connective_tissue_disease    0
peptic_ulcer_disease         2
diabetes                     0
mod_severe_ckd               2
hemiplegia                   0
leukemia                     0
malignant_lymphoma           0
solid_tumor                  0
liver_disease                1
aids                         0
cci_score                    5
respiratory_failure_type2    0
acute_renal_failure          0
dtype: int64

Saved patienthistory_clean.csv
